# YieldGuard — temperature scaling for WaferCNN

A softmax "0.97" is not a probability until it has been checked. **Temperature scaling**
(Guo et al., 2017) fits one number, **T**, that divides the logits before softmax:

- **T > 1** softens over-confident predictions; **T < 1** sharpens under-confident ones.
- On a single view it never changes which class wins, only how confident the model claims to be.

We serve the CNN with **TTA-8** (4 rotations × 2 mirrors, softmax averaged), so T is fitted for
exactly that pipeline: each view's logits are divided by T, then the 8 softmaxes are averaged.

**Honest split.** The shipped checkpoint was selected on the whole validation set. So validation
is split in half: **T is fitted on one half and judged on the other**, which neither the fit
nor anything else has touched.

**What this cannot do.** It fixes confidence on *the kinds of wafer the model has seen*. It does
not make the model recognise patterns outside its nine classes — cell 6 shows that directly.

Inference only, no training. About 10 minutes on a T4 (most of it the first data download).

In [ ]:
#@title 0. Settings
CKPT_URL = ("https://raw.githubusercontent.com/Parva-28/bob-ai-hackathon-DuckDuckGo/"
            "main/src/models/vision/checkpoints/best_model.pt")
CKPT_SHA = "47d5d5862c9548ce1ab87ce5d0531e722b0c0494111792a66d127cf8f7067954"  # holdout_results.json
CKPT     = "best_model.pt"
N_BINS   = 15        # bins for expected calibration error (ECE), the standard choice

In [ ]:
#@title 1. Install + Kaggle credentials
!pip -q install kagglehub

import os
from getpass import getpass
if not os.environ.get("KAGGLE_KEY"):
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
        print("using Colab Secrets")
    except Exception:
        os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ").strip()
        os.environ["KAGGLE_KEY"]      = getpass("Kaggle key: ").strip()
print("credentials set for:", os.environ.get("KAGGLE_USERNAME"))

In [ ]:
#@title 2. Data — same labelled split as train_simple.ipynb, validation cut in half
import numpy as np, sys, pickle, warnings
warnings.filterwarnings("ignore")
from PIL import Image
from sklearn.model_selection import train_test_split

CLASSES = ["Center","Donut","Edge-Loc","Edge-Ring","Local","Random","Scratch","Near-full","None"]
NC, CACHE = 9, "wm811k_64.npz"

if not os.path.exists(CACHE):          # identical to train_simple.ipynb, so the split matches
    import kagglehub, pandas as pd
    pkl = os.path.join(kagglehub.dataset_download("qingyi/wm811k-wafer-map"), "LSWMD.pkl")
    import pandas.core.indexes as _ci
    sys.modules.setdefault("pandas.indexes", _ci)
    sys.modules.setdefault("pandas.indexes.base", _ci.base)
    sys.modules.setdefault("pandas.indexes.numeric", getattr(_ci, "numeric", _ci.base))
    # pickle is WM-811K's own distribution format — the only way the dataset ships
    with open(pkl, "rb") as f:
        df = pickle.load(f, encoding="latin1")
    fix = {"none":"None","center":"Center","donut":"Donut","edge-loc":"Edge-Loc",
           "edge-ring":"Edge-Ring","loc":"Local","local":"Local","random":"Random",
           "scratch":"Scratch","near-full":"Near-full","nearfull":"Near-full"}
    def label_of(ft):
        if isinstance(ft, np.ndarray): ft = ft.tolist()
        if not ft or not ft[0]: return None
        s = str(ft[0][0] if isinstance(ft[0], list) else ft[0]).strip()
        return fix.get(s.lower(), s)
    df["label"] = [label_of(v) for v in df["failureType"]]
    lab = df[df["label"].notna()]
    none_m = lab["label"] == "None"
    use = pd.concat([lab[~none_m], lab[none_m].sample(frac=0.25, random_state=42)],
                    ignore_index=True)
    X, y = [], []
    for wm, lb in zip(use["waferMap"], use["label"]):
        if not isinstance(wm, np.ndarray) or wm.size == 0: continue
        X.append(np.array(Image.fromarray(wm.astype(np.uint8), "L")
                          .resize((64, 64), Image.NEAREST), dtype=np.uint8))
        y.append(CLASSES.index(lb))
    X = np.stack(X)[:, None]; y = np.array(y, dtype=np.int64)
    Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
    np.savez_compressed(CACHE, X_train=Xtr, y_train=ytr, X_val=Xva, y_val=yva)
    del df

d = np.load(CACHE)
X_val = d["X_val"].astype(np.float32) / 2.0       # {0,1,2} -> {0, 0.5, 1}
y_val = d["y_val"].astype(np.int64)

# Half A fits T. Half B judges it. Same split as self_training.ipynb.
fit_i, eval_i = train_test_split(np.arange(len(y_val)), test_size=0.5,
                                 stratify=y_val, random_state=42)
print(f"validation {len(y_val):,}  ->  fit half {len(fit_i):,}   evaluation half {len(eval_i):,}")

In [ ]:
#@title 3. Load the shipped checkpoint — fetched from the repo and verified by SHA-256
import hashlib, urllib.request, torch, torch.nn as nn, torch.nn.functional as F

if not os.path.exists(CKPT):
    urllib.request.urlretrieve(CKPT_URL, CKPT)
sha = hashlib.sha256(open(CKPT, "rb").read()).hexdigest()
assert sha == CKPT_SHA, f"checkpoint is not the shipped one: {sha[:16]}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False),
                                          nn.BatchNorm2d(cout))
    def forward(self, x):
        h = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(h)) + self.shortcut(x))

class WaferCNN(nn.Module):
    def __init__(self, nc=NC, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 16, 3, 1, 1, bias=False),
                                  nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        self.layer1 = ResBlock(16,  32,  stride=2)
        self.layer2 = ResBlock(32,  64,  stride=2)
        self.layer3 = ResBlock(64,  128, stride=2)
        self.layer4 = ResBlock(128, 128, stride=2)
        self.gap, self.dropout, self.fc = nn.AdaptiveAvgPool2d(1), nn.Dropout(dropout), nn.Linear(128, nc)
    def forward(self, x):
        x = self.layer4(self.layer3(self.layer2(self.layer1(self.stem(x)))))
        return self.fc(self.dropout(self.gap(x).flatten(1)))

model = WaferCNN().to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE, weights_only=True))  # strict names
model.eval()
print(f"loaded {CKPT}  sha256 {sha[:16]}…  on {DEVICE}")

In [ ]:
#@title 4. Logits for all 8 TTA views — computed once, reused for every T
VIEWS = [(k, f) for k in range(4) for f in (False, True)]     # 4 rotations x 2 mirrors

@torch.no_grad()
def view_logits(X, bs=2048):
    """(N, 8, NC) raw logits, one row per dihedral view."""
    T_ = torch.from_numpy(X); out = torch.zeros(len(X), len(VIEWS), NC)
    for vi, (k, flip) in enumerate(VIEWS):
        V = torch.rot90(T_, k, [-2, -1])
        if flip: V = V.flip(-1)
        for i in range(0, len(V), bs):
            out[i:i+bs, vi] = model(V[i:i+bs].to(DEVICE)).float().cpu()
    return out.numpy()

def probs_at(L, T):
    """The served pipeline: divide each view's logits by T, softmax, average the 8 views."""
    z = L / T
    z = z - z.max(-1, keepdims=True)                  # numerically stable softmax
    e = np.exp(z)
    return (e / e.sum(-1, keepdims=True)).mean(1)

L_val = view_logits(X_val)
LA, yA = L_val[fit_i],  y_val[fit_i]
LB, yB = L_val[eval_i], y_val[eval_i]
print("logits:", L_val.shape)

In [ ]:
#@title 5. Fit T on half A, judge it on half B
def nll(p, y):
    return -np.log(np.clip(p[np.arange(len(y)), y], 1e-12, None)).mean()

def ece(p, y, bins=N_BINS):
    """Expected calibration error: average gap between confidence and accuracy, per bin."""
    conf, hit = p.max(1), (p.argmax(1) == y)
    edges, total = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): total += m.mean() * abs(hit[m].mean() - conf[m].mean())
    return total

def brier(p, y):
    oh = np.eye(NC)[y]
    return ((p - oh) ** 2).sum(1).mean()

# One parameter, so a fine grid is exact enough and cannot get stuck the way an optimiser can.
grid = np.round(np.arange(0.25, 5.0001, 0.01), 2)
fit_nll = np.array([nll(probs_at(LA, t), yA) for t in grid])
T = float(grid[fit_nll.argmin()])
assert 0.25 < T < 5.0, f"T={T} sits on the edge of the search grid — widen the grid"

p1, pT = probs_at(LB, 1.0), probs_at(LB, T)
rows = [("ECE (lower is better)",    ece(p1, yB),   ece(pT, yB)),
        ("NLL (lower is better)",    nll(p1, yB),   nll(pT, yB)),
        ("Brier (lower is better)",  brier(p1, yB), brier(pT, yB)),
        ("mean confidence",          p1.max(1).mean(), pT.max(1).mean()),
        ("accuracy",                 (p1.argmax(1) == yB).mean(), (pT.argmax(1) == yB).mean())]
print(f"fitted on half A:  T = {T:.2f}   "
      f"({'softens over-confidence' if T > 1 else 'sharpens under-confidence' if T < 1 else 'no change'})\n")
print(f"{'judged on half B':<26}{'T = 1':>10}{f'T = {T:.2f}':>12}")
for name, a, b in rows:
    print(f"{name:<26}{a:>10.4f}{b:>12.4f}")

changed = (p1.argmax(1) != pT.argmax(1)).sum()
print(f"\npredictions changed by T: {changed} of {len(yB):,}"
      "  (T can only flip a TTA vote when the 8 views disagree)")

# reliability diagram
import matplotlib.pyplot as plt
def reliability(p, y, ax, title):
    conf, hit = p.max(1), (p.argmax(1) == y)
    edges = np.linspace(0, 1, N_BINS + 1); mids, accs, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= 5:
            mids.append(conf[m].mean()); accs.append(hit[m].mean()); ns.append(m.sum())
    ax.plot([0, 1], [0, 1], ls="--", c="#6a7d8c", lw=1, label="perfectly calibrated")
    ax.plot(mids, accs, "o-", c="#ca3b37", lw=1.5, label="WaferCNN")
    for x_, y_, n_ in zip(mids, accs, ns):
        ax.annotate(f"{n_}", (x_, y_), textcoords="offset points", xytext=(0, 6), fontsize=6, ha="center")
    ax.set(xlim=(0, 1), ylim=(0, 1.02), xlabel="stated confidence", ylabel="actual accuracy",
           title=f"{title}   ECE {ece(p, y):.4f}")
    ax.legend(fontsize=7, loc="upper left"); ax.grid(alpha=.25)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
reliability(p1, yB, axes[0], "before (T = 1)")
reliability(pT, yB, axes[1], f"after (T = {T:.2f})")
fig.suptitle("Evaluation half only. Below the diagonal = over-confident. "
             "Numbers are wafers per bin.", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
#@title 6. The limit: patterns the model has never seen
# Temperature scaling is fitted on wafers like the training data. It does not teach the model
# to say "none of these" — so check what it does to inputs outside the nine classes.
Y, X_ = np.mgrid[0:64, 0:64]
disc = np.hypot(Y - 31.5, X_ - 31.5) <= 30.0
def wafer(fail):
    a = np.zeros((64, 64)); a[disc] = 1; a[disc & fail] = 2; return a
cases = {
    "empty grid (no wafer at all)":         np.zeros((64, 64)),
    "row-pair stripes (test artifact)":     wafer(((Y // 3) % 2 == 0)),
    "every die failed except 3 bands":      wafer(~np.isin(Y, [16, 32, 48])),
}
print(f"{'input':<36}{'class':>11}{'T = 1':>8}{f'T = {T:.2f}':>10}")
for name, a in cases.items():
    Lx = view_logits((a / 2.0).astype(np.float32)[None, None])
    q1, qT = probs_at(Lx, 1.0)[0], probs_at(Lx, T)[0]
    print(f"{name:<36}{CLASSES[q1.argmax()]:>11}{q1.max():>8.2f}{qT.max():>10.2f}")
print("\nIf these stay confident, that is the expected result: calibration fixes confidence on\n"
      "familiar wafers. Unfamiliar ones still need the validation gate that runs before the model.")

In [ ]:
#@title 7. Save temperature.json (+ download)
import json
json.dump({
    "temperature": T,
    "applies_to": "TTA-8: each view's logits are divided by T before softmax, then the 8 are averaged",
    "fitted_on": f"WM-811K validation half A (n={len(yA)}), selected by NLL over T in [0.25, 5.0]",
    "evaluated_on": f"WM-811K validation half B (n={len(yB)}), not used for the fit",
    "checkpoint_sha256": sha,
    "before": {"ece": round(float(ece(p1, yB)), 4), "nll": round(float(nll(p1, yB)), 4),
               "brier": round(float(brier(p1, yB)), 4),
               "mean_confidence": round(float(p1.max(1).mean()), 4),
               "accuracy": round(float((p1.argmax(1) == yB).mean()), 4)},
    "after":  {"ece": round(float(ece(pT, yB)), 4), "nll": round(float(nll(pT, yB)), 4),
               "brier": round(float(brier(pT, yB)), 4),
               "mean_confidence": round(float(pT.max(1).mean()), 4),
               "accuracy": round(float((pT.argmax(1) == yB).mean()), 4)},
    "notes": ["Calibrates confidence on in-distribution wafers only. It does not detect "
              "out-of-distribution inputs; the validation gate before the model does that.",
              "The checkpoint was selected on the full validation set, so half B is clean for "
              "the fit of T but not fully independent of model selection."],
}, open("temperature.json", "w"), indent=2)
print(open("temperature.json").read())

try:
    from google.colab import files
    files.download("temperature.json")
except ImportError:
    pass

## Reading the result

- **T > 1** means the model was over-confident (the usual finding for modern CNNs);
  **T < 1** means TTA averaging had already made it under-confident.
- **ECE is the headline.** It is the average gap between "how sure the model says it is" and
  "how often it is right". A drop from, say, 0.03 to 0.01 means a stated 0.90 now really means
  about 90%.
- **Accuracy should barely move.** If it does, look at "predictions changed": T only flips a
  TTA vote when the 8 views disagree.
- **Cell 6 is the honest limit.** Calibration makes familiar wafers' confidence trustworthy.
  It does nothing for wafers outside the nine classes, which is why the validation gate exists.

## Using it in the repo

Put the file at `src/models/vision/temperature.json`. The classifier then divides each view's
logits by `temperature` before the softmax in its TTA loop. That one division is the whole
change.